## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-11-15-49-55 +0000,nyt,Workers in Spain Stage a General Strike for th...,https://www.nytimes.com/2026/11/11/world/europ...
1,2026-11-11-15-47-21 +0000,bbc,Burnham signals backing for more ethnicity opt...,https://www.bbc.co.uk/news/articles/c8dx160v9z...
2,2026-11-11-15-46-03 +0000,nypost,‘Cowboy’ chef serves up hope to ensure America...,https://nypost.com/2026/11/11/us-news/cowboy-c...
3,2026-11-11-15-45-32 +0000,chitri,Skokie advances 58-unit townhome project with ...,https://www.chicagotribune.com/2026/11/11/skok...
4,2026-11-11-15-45-00 +0000,wapo,Victorious Democrats promise to cut costs — bu...,https://www.washingtonpost.com/politics/2026/1...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2694/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
83,trump,32
192,nvidia,23
74,huang,21
858,new,17
201,spy,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
256,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...,153
384,2026-11-10-19-21-16 +0000,nypost,Nvidia loses nearly $450B in two-day wipeout a...,https://nypost.com/2026/11/10/business/nvidia-...,141
22,2026-11-11-15-04-13 +0000,nypost,Nvidia CEO Jensen Huang tries to act ‘business...,https://nypost.com/2026/11/11/business/nvidias...,134
358,2026-11-10-20-22-14 +0000,nypost,Jensen Huang’s ties to suspected Chinese spy h...,https://nypost.com/2026/11/10/business/jensen-...,130
230,2026-11-11-02-00-00 +0000,wsj,Lawmakers are demanding tighter restrictions o...,https://www.wsj.com/politics/policy/jensen-hua...,127


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
256,153,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...
100,115,2026-11-11-11-53-00 +0000,wsj,House Republicans saved Pete Hegseth and picke...,https://www.wsj.com/politics/policy/house-gop-...
249,72,2026-11-11-00-29-45 +0000,latimes,Meghan Markle's father Thomas Markle Sr. cut o...,https://www.latimes.com/entertainment-arts/sto...
301,70,2026-11-10-22-30-00 +0000,wsj,"Senate Majority Leader John Thune (R., S.D.) w...",https://www.wsj.com/politics/policy/john-thune...
424,58,2026-11-10-17-36-51 +0000,nypost,Gas prices drop below $4 a gallon for first ti...,https://nypost.com/2026/11/10/business/gas-pri...
20,56,2026-11-11-15-05-11 +0000,bbc,Russia-Ukraine talks turn to post-war security...,https://www.bbc.co.uk/news/articles/ck792we7zr...
248,53,2026-11-11-00-30-47 +0000,chitri,Veteran Business Project helps ex-military car...,https://www.chicagotribune.com/2026/11/10/vete...
401,50,2026-11-10-18-35-35 +0000,bbc,"'If I tell you to run, run': BBC joins a Ukrai...",https://www.bbc.co.uk/news/videos/cm2l4l59gjp7...
428,49,2026-11-10-17-30-00 +0000,wapo,Trump-appointed Fed member signals big interes...,https://www.washingtonpost.com/business/2026/1...
225,46,2026-11-11-02-09-00 +0000,wsj,The Kennedy family is joining the efforts to b...,https://www.wsj.com/politics/policy/kennedy-fa...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
